# Project AI Programming foundations
Name: Ronald B.

Dataset: COIL 2000

This project is about building a reproducible data workflow using the COIL 2000 dataset. The dataset contains real customer data from a Dutch insurance company, it has 5,822 customers and 86 columns. In this notebook I load, clean, explore and visualize the data to find out which customer characteristics are linked to owning a caravan insurance policy. 

## Setup

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Ingestion

In [2]:
# Read the 86 column names from the data dictionary
column_names = []

with open("dictionary.txt", encoding="latin-1") as f:
    for line in f:
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit() and parts[1].isupper():
            column_names.append(parts[1])
        if len(column_names) == 86:
            break

# Load the dataset
df = pd.read_csv("ticdata2000.txt", sep="\t", header=None, names=column_names)

## First look at the data

In [3]:
# Check dataframe
print(df.shape)
df.head()

(5822, 86)


,MOSTYPE,MAANTHUI,MGEMOMV,MGEMLEEF,MOSHOOFD,MGODRK,MGODPR,MGODOV,MGODGE,MRELGE,...,APERSONG,AGEZONG,AWAOREG,ABRAND,AZEILPL,APLEZIER,AFIETS,AINBOED,ABYSTAND,CARAVAN
0,33,1,3,2,8,0,5,1,3,7,...,0,0,0,1,0,0,0,0,0,0
1,37,1,2,2,8,1,4,1,4,6,...,0,0,0,1,0,0,0,0,0,0
2,37,1,2,2,8,0,4,2,4,3,...,0,0,0,1,0,0,0,0,0,0
3,9,1,3,3,3,2,3,2,4,5,...,0,0,0,1,0,0,0,0,0,0
4,40,1,4,2,10,1,4,1,4,7,...,0,0,0,1,0,0,0,0,0,0


In [4]:
# Column types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5822 entries, 0 to 5821
Data columns (total 86 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   MOSTYPE   5822 non-null   int64
 1   MAANTHUI  5822 non-null   int64
 2   MGEMOMV   5822 non-null   int64
 3   MGEMLEEF  5822 non-null   int64
 4   MOSHOOFD  5822 non-null   int64
 5   MGODRK    5822 non-null   int64
 6   MGODPR    5822 non-null   int64
 7   MGODOV    5822 non-null   int64
 8   MGODGE    5822 non-null   int64
 9   MRELGE    5822 non-null   int64
 10  MRELSA    5822 non-null   int64
 11  MRELOV    5822 non-null   int64
 12  MFALLEEN  5822 non-null   int64
 13  MFGEKIND  5822 non-null   int64
 14  MFWEKIND  5822 non-null   int64
 15  MOPLHOOG  5822 non-null   int64
 16  MOPLMIDD  5822 non-null   int64
 17  MOPLLAAG  5822 non-null   int64
 18  MBERHOOG  5822 non-null   int64
 19  MBERZELF  5822 non-null   int64
 20  MBERBOER  5822 non-null   int64
 21  MBERMIDD  5822 non-null   int64
 22  MBERARBG  5

In [5]:
# Missing values in the whole dataset
df.isna().sum().sum()

np.int64(0)

No missing values in the dataset, so we don't need to replace missing values with imputation.

In [6]:
# Duplicate rows
df.duplicated().sum()

np.int64(602)

602 duplicate rows, about 10% of the data. 

In [7]:
# Summary statistics, flipped so each column is a row
df.describe().T

,count,mean,std,min,25%,50%,75%,max
MOSTYPE,5822.0,24.253349,12.846706,1.0,10.0,30.0,35.0,41.0
MAANTHUI,5822.0,1.110615,0.405842,1.0,1.0,1.0,1.0,10.0
MGEMOMV,5822.0,2.678805,0.789835,1.0,2.0,3.0,3.0,5.0
MGEMLEEF,5822.0,2.991240,0.814589,1.0,2.0,3.0,3.0,6.0
MOSHOOFD,5822.0,5.773617,2.856760,1.0,3.0,7.0,8.0,10.0
...,...,...,...,...,...,...,...,...
APLEZIER,5822.0,0.006012,0.081632,0.0,0.0,0.0,0.0,2.0
AFIETS,5822.0,0.031776,0.210986,0.0,0.0,0.0,0.0,3.0
AINBOED,5822.0,0.007901,0.090463,0.0,0.0,0.0,0.0,2.0
ABYSTAND,5822.0,0.014256,0.119996,0.0,0.0,0.0,0.0,2.0


In [8]:
# Different values for each column
df.nunique()

MOSTYPE     40
MAANTHUI     9
MGEMOMV      5
MGEMLEEF     6
MOSHOOFD    10
            ..
APLEZIER     3
AFIETS       4
AINBOED      3
ABYSTAND     3
CARAVAN      2
Length: 86, dtype: int64

In [9]:
# How many customers have a caravan policy
print(df["CARAVAN"].value_counts())
print(df["CARAVAN"].value_counts(normalize=True))

CARAVAN
0    5474
1     348
Name: count, dtype: int64
CARAVAN
0    0.940227
1    0.059773
Name: proportion, dtype: float64


Only 6% of the customers have a caravan policy, so the two groups are very unbalanced.

## Cleaning

The 602 duplicate rows will not be cleaned, because the area columns are shared by everyone with the same postal code, two separate real customers can have identical rows. Removing them would mean that real customers would be removed, leaving us with less data. 

In [ ]:
# Rename columns function

def rename_columns(df):
    """
    Renaming the most important columns with readable English names.

    The column names are short Dutch codes, like MOSHOOFD, which are hard to read. 
    The new names are based on the data dictionary.
    Returns a new dataframe, the original dataframe is not changed.
    """
    new_names = {
        "MOSHOOFD": "customer_main_type",
        "MGEMLEEF": "age_band",
        "MAANTHUI": "num_houses",
        "MGEMOMV": "household_size",
        "MHKOOP": "home_owners",
        "MINKGEM": "avg_income",
        "MKOOPKLA": "purchasing_power",
        "PPERSAUT": "car_policy_contribution",
        "APERSAUT": "num_car_policies",
        "PBRAND": "fire_policy_contribution",
        "CARAVAN": "caravan_policy",
    }
    return df.rename(columns=new_names)

In [ ]:
# Decode categories function

def decode_categories(df):
    """
    Replacing the number codes with readable labels for age band and customer type.

    In the original data these columns only contain codes like age_band 3 means 40-50 years.
    The labels come from tables L1 and L2 in the data dictionary. 
    Returns a new dataframe, the original dataframe is not changed.
    """
    age_labels = {1: "20-30", 2: "30-40", 3: "40-50",
                  4: "50-60", 5: "60-70", 6: "70-80"}
    main_type_labels = {
        1: "Successful hedonists", 2: "Driven growers", 3: "Average family",
        4: "Career loners", 5: "Living well", 6: "Cruising seniors",
        7: "Retired and religious", 8: "Family with grown ups",
        9: "Conservative families", 10: "Farmers",
    }

    df = df.copy()
    df["age_band"] = df["age_band"].map(age_labels)
    df["customer_main_type"] = df["customer_main_type"].map(main_type_labels)
    return df

In [12]:
# Creating a clean dataframe by using the functions rename_columns and decode_categories

df_clean = rename_columns(df)
df_clean = decode_categories(df_clean)

In [13]:
# Check that every code has a label

print(df_clean[["age_band","customer_main_type"]].isna().sum())
df_clean.head()

age_band              0
customer_main_type    0
dtype: int64


,MOSTYPE,num_houses,household_size,age_band,customer_main_type,MGODRK,MGODPR,MGODOV,MGODGE,MRELGE,...,APERSONG,AGEZONG,AWAOREG,ABRAND,AZEILPL,APLEZIER,AFIETS,AINBOED,ABYSTAND,caravan_policy
0,33,1,3,30-40,Family with grown ups,0,5,1,3,7,...,0,0,0,1,0,0,0,0,0,0
1,37,1,2,30-40,Family with grown ups,1,4,1,4,6,...,0,0,0,1,0,0,0,0,0,0
2,37,1,2,30-40,Family with grown ups,0,4,2,4,3,...,0,0,0,1,0,0,0,0,0,0
3,9,1,3,40-50,Average family,2,3,2,4,5,...,0,0,0,1,0,0,0,0,0,0
4,40,1,4,30-40,Farmers,1,4,1,4,7,...,0,0,0,1,0,0,0,0,0,0
